# Semantic Search & LLM Application

Prototype asisten customer service e-commerce: FAQ knowledge base → Gemini embeddings → FAISS → semantic search → prompt terstruktur → jawaban Gemini.

Alur: **Ingest → Embed → Store → Retrieve → Generate**.

## 1. Import Library & Load Knowledge Base

In [ ]:
import os
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
load_dotenv(ROOT / '.env')
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

EMBED_MODEL = 'gemini-embedding-001'
LLM_MODEL = 'gemini-3.8-flash'

df = pd.read_csv(ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

In [ ]:
print('Kolom:', list(df.columns))
print('Missing value:', df.isna().sum().sum())
print('faq_id unik:', df['faq_id'].is_unique)
print()
print(df['category'].value_counts().to_string())

In [ ]:
df['answer_words'] = df['answer'].str.split().str.len()
df['answer_words'].describe()

Knowledge base berisi 59 entri dalam 8 kategori. Jawaban rata-rata pendek (1-3 kalimat), jadi satu entri FAQ sudah cukup kecil untuk menjadi satu unit retrieval tanpa chunking.

## 2. Menyiapkan Unit Retrievable

Question dan answer digabung menjadi satu teks per entri. Kategori ikut disertakan karena memberi sinyal topik tambahan (misal "Pengembalian" untuk query soal refund). Data asli di `data/` tidak diubah; kolom `content` hanya ada di memori.

In [ ]:
df['content'] = (
    'Kategori: ' + df['category']
    + '\nPertanyaan: ' + df['question']
    + '\nJawaban: ' + df['answer']
)
documents = df['content'].tolist()
print('Contoh unit retrievable:\n')
print(documents[0])

## 3. Membuat Embeddings

Model `gemini-embedding-001` (`text-embedding-004` pada starter sudah tidak tersedia). Dokumen di-embed dengan `task_type=RETRIEVAL_DOCUMENT` dan query dengan `RETRIEVAL_QUERY`, sesuai rekomendasi Gemini untuk skenario asymmetric retrieval. Vektor dinormalisasi L2 agar inner product = cosine similarity.

In [ ]:
# Free tier Gemini membatasi request per menit; retry dengan jeda saat kena 429/5xx.
def with_retry(fn, retries=4):
    for attempt in range(retries):
        try:
            return fn()
        except genai.errors.APIError as err:
            if attempt == retries - 1 or err.code not in (429, 500, 503):
                raise
            time.sleep(20 * (attempt + 1))


def embed(texts, task_type):
    result = with_retry(lambda: client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    ))
    vectors = np.array([e.values for e in result.embeddings], dtype='float32')
    faiss.normalize_L2(vectors)
    return vectors


embeddings = embed(documents, 'RETRIEVAL_DOCUMENT')
print('Shape embeddings:', embeddings.shape)
print('Norm vektor pertama:', np.linalg.norm(embeddings[0]).round(4))
assert embeddings.shape[0] == len(df)

## 4. Menyimpan Embeddings ke Vector Database (FAISS)

`IndexFlatIP` (exact search, inner product). Dengan vektor ternormalisasi, skornya adalah cosine similarity: makin tinggi makin relevan. Untuk 59 vektor, exact search sudah instan sehingga index aproksimasi (IVF/HNSW) tidak diperlukan.

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print('Dimensi:', dimension)
print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(df)

index_path = ROOT / 'index' / 'faq_index.faiss'
index_path.parent.mkdir(exist_ok=True)
faiss.write_index(index, str(index_path))

loaded = faiss.read_index(str(index_path))
print('Index tersimpan di', index_path.relative_to(ROOT), '| ntotal setelah load:', loaded.ntotal)

## 5. Mengimplementasikan Semantic Search

In [ ]:
def semantic_search(query, top_k=3):
    query_vector = embed([query], 'RETRIEVAL_QUERY')
    scores, indices = index.search(query_vector, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        row = df.iloc[idx]
        results.append({
            'rank': rank,
            'faq_id': int(row['faq_id']),
            'category': row['category'],
            'question': row['question'],
            'answer': row['answer'],
            'score': round(float(score), 4),
        })
    return results


def show(results):
    return pd.DataFrame(results)[['rank', 'faq_id', 'category', 'question', 'score']]


show(semantic_search('kenapa paket saya belum sampai?'))

## 6. Mendesain Prompt dan Context

Prompt dipisah menjadi empat bagian eksplisit:

- **Persona**: agen CS e-commerce yang ramah dan lugas.
- **Task**: jawab hanya dari context, akui jika informasi tidak ada, jangan mengarang kebijakan.
- **Context**: top-k FAQ hasil retrieval beserta `faq_id` dan skor similarity.
- **Format**: Bahasa Indonesia, singkat, langkah bernomor jika prosedural, sebut sumber FAQ.

Skor similarity ikut ditampilkan di context agar LLM tahu bahwa entri dengan skor rendah mungkin tidak relevan dan tidak memaksakan jawaban dari situ.

In [ ]:
PROMPT_TEMPLATE = '''# PERSONA
Kamu adalah agen customer service sebuah platform e-commerce. Gaya bicaramu ramah, sopan, dan to the point.

# TASK
Jawab pertanyaan pelanggan HANYA berdasarkan entri FAQ pada bagian CONTEXT.
- Gunakan hanya entri FAQ yang benar-benar menjawab pertanyaan; abaikan entri yang tidak relevan walaupun ikut muncul di CONTEXT.
- Jangan menambahkan kebijakan, angka, nama layanan, atau prosedur yang tidak tertulis di CONTEXT, dan jangan memakai pengetahuan umum di luar CONTEXT.
- Jika CONTEXT tidak memuat jawaban, katakan dengan jelas bahwa informasi tersebut belum tersedia di FAQ, lalu sarankan pelanggan menghubungi customer service.
- Jika CONTEXT hanya menjawab sebagian, jawab bagian yang tersedia dan sebutkan bagian mana yang tidak tercakup.

# CONTEXT
{context}

# FORMAT
- Bahasa Indonesia, maksimal 4 kalimat atau 5 poin.
- Jika jawabannya berupa prosedur, tulis sebagai langkah bernomor.
- Akhiri dengan satu baris: "Sumber: FAQ #<id>" (pisahkan dengan koma jika lebih dari satu), atau "Sumber: -" jika tidak ada FAQ yang relevan.

# PERTANYAAN PELANGGAN
{query}
'''


def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ #{r['faq_id']} | {r['category']} | similarity {r['score']:.3f}]\n"
        f"Pertanyaan: {r['question']}\nJawaban: {r['answer']}"
        for r in search_results
    )
    return PROMPT_TEMPLATE.format(context=context, query=query)


print(build_prompt('apakah bisa bayar pakai cicilan?', semantic_search('apakah bisa bayar pakai cicilan?')))

## 7. Mengintegrasikan LLM

`generate_answer` menyatukan seluruh alur: query → retrieval → prompt → jawaban. Temperature rendah (0.2) dipakai karena jawaban CS harus konsisten dan setia pada FAQ, bukan kreatif.

In [ ]:
def call_llm(prompt):
    response = with_retry(lambda: client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2),
    ))
    return response.text.strip()


def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    answer = call_llm(build_prompt(query, search_results))
    return answer, search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:\n')
print(answer)
show(sources)

## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Tujuh query dibagi tiga kelompok. Kolom `expected` berisi `faq_id` yang seharusnya ter-retrieve (ditentukan manual dengan membaca knowledge base); `None` berarti informasinya memang tidak ada di FAQ.

In [ ]:
test_cases = [
    # Ada jawabannya secara langsung
    {'type': 'direct', 'query': 'Bagaimana cara membatalkan pesanan yang sudah dibuat?', 'expected': 3},
    {'type': 'direct', 'query': 'Metode pembayaran apa saja yang tersedia?', 'expected': 17},
    # Parafrase / sinonim / bahasa informal
    {'type': 'paraphrase', 'query': 'aku lupa kata sandi, gimana caranya bisa masuk lagi ke akunku?', 'expected': 32},
    {'type': 'paraphrase', 'query': 'saldo m-banking udah kepotong tapi orderan masih nunggu dibayar, kok gitu?', 'expected': 21},
    {'type': 'paraphrase', 'query': 'barang yang dateng pecah dan penyok, harus ngapain?', 'expected': 16},
    # Tidak ada di knowledge base
    {'type': 'out-of-kb', 'query': 'Apakah ada toko offline di Surabaya untuk ambil barang langsung?', 'expected': None},
    {'type': 'out-of-kb', 'query': 'Bisa kirim paket ke luar negeri, misalnya ke Singapura?', 'expected': None},
]

rows = []
for case in test_cases:
    answer, sources = generate_answer(case['query'])
    retrieved = [s['faq_id'] for s in sources]
    rows.append({
        'type': case['type'],
        'query': case['query'],
        'expected': case['expected'],
        'top3_ids': retrieved,
        'top3_scores': [s['score'] for s in sources],
        'hit@1': None if case['expected'] is None else retrieved[0] == case['expected'],
        'hit@3': None if case['expected'] is None else case['expected'] in retrieved,
        'answer': answer,
    })

    print(f"[{case['type']}] {case['query']}")
    for s in sources:
        print(f"  #{s['faq_id']:>2} ({s['score']:.3f}) {s['question']}")
    print('\nJawaban:\n' + answer)
    print('-' * 90)

results = pd.DataFrame(rows)

In [ ]:
results[['type', 'query', 'expected', 'top3_ids', 'top3_scores', 'hit@1', 'hit@3']]

In [ ]:
in_kb = results[results['expected'].notna()]
print(f"Hit@1 (query in-KB): {in_kb['hit@1'].mean():.0%}")
print(f"Hit@3 (query in-KB): {in_kb['hit@3'].mean():.0%}")
print()
print('Skor top-1 rata-rata per tipe query:')
print(results.assign(top1=results['top3_scores'].str[0]).groupby('type', sort=False)['top1'].agg(['mean', 'min', 'max']).round(3))

In [ ]:
results.to_csv(ROOT / 'documentation' / 'retrieval_test_results.csv', index=False)

### Catatan Analisis Relevansi

ANALYSIS_PLACEHOLDER